# Template · Markov network (pgmpy + brute-force check)
Replace the edges and the potential. For ≤ 6 binary/ternary variables always check by enumeration.

# Markov Networks – Quick answers

**Q: What is a Markov network?**
A: An undirected graphical model. Nodes are random variables and edges mean direct interaction (no causal direction). The joint distribution is the normalised product of the clique potentials: P(x) = (1/Z) ∏_C φ_C(x_C).

**Q: What is a clique / maximal clique?**
A: A clique is a set of nodes where every pair is connected. A maximal clique cannot be extended with another node. Found with `nx.find_cliques(G)`.

**Q: What is a potential φ?**
A: A non-negative function on a clique that measures how compatible its values are. It is not a probability and does not need to sum to 1.

**Q: What is Z (partition function)?**
A: The sum of the product of potentials over all configurations. Dividing by Z normalises the distribution so it sums to 1.

**Q: When is A ⊥ C | B?**
A: When every path from A to C passes through B (graph separation). Unlike d-separation, there are no colliders: observing a node always blocks the path. Check: remove B from the graph and test `nx.has_path(H, "A", "C")`.

**Q: What is the energy form?**
A: φ = exp(−E), so P(x) ∝ exp(−E(x)). Lower energy means higher probability. The MAP configuration is the one with minimum energy.

**Q: What is the Ising model?**
A: Variables take values −1 or +1. The energy E(s) = −J Σ_(i,j) s_i s_j − h Σ_i s_i rewards neighbours with equal values, so all −1 and all +1 are the most probable configurations.

**Q: What is the MAP configuration?**
A: The single configuration with the highest joint probability. It can differ from taking the most probable value of each variable separately.

**Q: Which pgmpy classes are used?**
A: `DiscreteMarkovNetwork` (the graph), `DiscreteFactor` (the potentials) and `BeliefPropagation` (inference: `query` for marginals, `map_query` for the MAP).

In [ ]:
%pip install -q pgmpy networkx matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product
from pgmpy.models import DiscreteMarkovNetwork
from pgmpy.factors.discrete import DiscreteFactor
from pgmpy.inference import VariableElimination

In [ ]:
edges = [("A", "B"), ("B", "C"), ("A", "C"), ("C", "D")]
G = nx.Graph(edges)
print("Maximal cliques:", [sorted(c) for c in nx.find_cliques(G)])
nx.draw(G, with_labels=True, node_size=1200, node_color="#ead1dc", pos=nx.spring_layout(G, seed=1))
plt.savefig("markov_network.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
beta = 1.0
mrf = DiscreteMarkovNetwork(edges)
for u, v in edges:
    # values in C order (first variable changes SLOWEST): phi(0,0), phi(0,1), phi(1,0), phi(1,1)
    values = [np.exp(beta * a * b) for a, b in product([0, 1], repeat=2)]
    mrf.add_factors(DiscreteFactor([u, v], [2, 2], values))
print("Model OK:", mrf.check_model())

ve = VariableElimination(mrf)
q = ve.query(["A"], evidence={"D": 1}, show_progress=False)
q.normalize()                         # potentials are not probabilities -> normalise
print(q)
print("MAP:", ve.map_query(["A", "B", "C"], evidence={"D": 1}, show_progress=False))

In [ ]:
# Brute force: P(x) = product of potentials / Z
nodes = ["A", "B", "C", "D"]
idx = {n: i for i, n in enumerate(nodes)}
configs = np.array(list(product([0, 1], repeat=len(nodes))))
score = np.exp(beta * sum(configs[:, idx[u]] * configs[:, idx[v]] for u, v in edges))
P = score / score.sum()
print("Z =", round(score.sum(), 4), "  most probable:", configs[P.argmax()], round(P.max(), 4))